# 📊 Task 2: Exploratory Data Analysis (EDA)

## Objective

The objective of this task is to clean, preprocess, and analyze the scraped book dataset to understand its characteristics and identify useful business insights.

## Libraries Used

- Pandas
- NumPy
- SciPy

## Data Cleaning

The following preprocessing steps were performed:

- Removed missing values
- Removed duplicate records
- Converted data types
- Created Price Bands
- Identified In-Stock books

## Statistical Analysis

The following analyses were performed:

- Summary Statistics
- Missing Value Analysis
- Duplicate Detection
- Rating Distribution
- Price Distribution
- Pearson Correlation Test
- Outlier Detection (IQR Method)

## Business Questions

- What is the average book price?
- Which rating appears most frequently?
- Does a higher rating indicate a higher price?
- Which price category contains the most books?
- Are there any price outliers?

## Output

- `books_clean.csv`

The cleaned dataset is used in the visualization task.

In [6]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv("data/raw/books_dataset.csv", encoding="utf-8-sig")

# --- Structure ---
print(df.shape)
print(df.dtypes)
df.head()

(1000, 5)
Title               str
Price           float64
Availability        str
Rating            int64
DetailURL           str
dtype: object


,Title,Price,Availability,Rating,DetailURL
0,A Light in the Attic,51.77,In stock,3,https://books.toscrape.com/catalogue/a-light-i...
1,Tipping the Velvet,53.74,In stock,1,https://books.toscrape.com/catalogue/tipping-t...
2,Soumission,50.10,In stock,1,https://books.toscrape.com/catalogue/soumissio...
3,Sharp Objects,47.82,In stock,4,https://books.toscrape.com/catalogue/sharp-obj...
4,Sapiens: A Brief History of Humankind,54.23,In stock,5,https://books.toscrape.com/catalogue/sapiens-a...


In [7]:
print("="*60)
print("DATASET INFORMATION")
print("="*60)

df.info()

DATASET INFORMATION
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Title         1000 non-null   str    
 1   Price         1000 non-null   float64
 2   Availability  1000 non-null   str    
 3   Rating        1000 non-null   int64  
 4   DetailURL     1000 non-null   str    
dtypes: float64(1), int64(1), str(3)
memory usage: 172.3 KB


In [9]:
# --- Cleaning ---
df["Price"] = pd.to_numeric(df["Price"], errors="coerce")
df["Rating"] = pd.to_numeric(df["Rating"], errors="coerce").astype("Int64")

df = df.dropna(subset=["Title", "Price"])
df = df.drop_duplicates(subset=["Title", "Price"])

df["InStock"] = df["Availability"].str.contains("In stock", case=False, na=False)
df["PriceBand"] = pd.cut(
    df["Price"],
    bins=[0, 20, 35, 50, np.inf],
    labels=["Budget (<£20)", "Mid (£20-35)", "Premium (£35-50)", "Luxury (£50+)"],
)

df.to_csv("data/processed/books_clean.csv", index=False, encoding="utf-8-sig")
df.describe()

,Price,Rating
count,1000.00000,1000.0
mean,35.07035,2.923
std,14.44669,1.434967
min,10.00000,1.0
25%,22.10750,2.0
50%,35.98000,3.0
75%,47.45750,4.0
max,59.99000,5.0


In [10]:
print(df.describe(include="object"))

                         Title Availability  \
count                     1000         1000   
unique                     999            1   
top     The Star-Touched Queen     In stock   
freq                         2         1000   

                                                DetailURL  
count                                                1000  
unique                                               1000  
top     https://books.toscrape.com/catalogue/a-light-i...  
freq                                                    1  


C:\Users\Jawairia Yousaf\AppData\Local\Temp\ipykernel_13140\1129007761.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  print(df.describe(include="object"))


In [11]:
# --- Missing values ---
print(df.isna().sum())

Title           0
Price           0
Availability    0
Rating          0
DetailURL       0
InStock         0
PriceBand       0
dtype: int64


In [12]:
print("Duplicate Records:", df.duplicated().sum())

Duplicate Records: 0


In [13]:
# --- Rating distribution ---
print(df["Rating"].value_counts().sort_index())
print(df["Availability"].value_counts())

Rating
1    226
2    196
3    203
4    179
5    196
Name: count, dtype: Int64
Availability
In stock    1000
Name: count, dtype: int64


In [14]:
print(df["Availability"].value_counts())
print(df["Rating"].value_counts().sort_index())

Availability
In stock    1000
Name: count, dtype: int64
Rating
1    226
2    196
3    203
4    179
5    196
Name: count, dtype: Int64


In [15]:
# --- Hypothesis test: does rating correlate with price? ---
valid = df.dropna(subset=["Rating", "Price"])
corr, p_value = stats.pearsonr(valid["Rating"].astype(float), valid["Price"])

print(f"Pearson correlation: {corr:.3f}")
print(f"p-value: {p_value:.4f}")
print("Significant" if p_value < 0.05 else "Not significant", "at α = 0.05")

Pearson correlation: 0.028
p-value: 0.3736
Not significant at α = 0.05


In [16]:
# --- Outlier detection (IQR method) ---
q1, q3 = df["Price"].quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outliers = df[(df["Price"] < lower) | (df["Price"] > upper)]
print(f"IQR bounds: [{lower:.2f}, {upper:.2f}]")
print(f"Outliers found: {len(outliers)}")
outliers[["Title", "Price"]]

IQR bounds: [-15.92, 85.48]
Outliers found: 0


,Title,Price


In [17]:
## ---Price Analysis by Rating---
print(
    df.groupby("Rating")["Price"]
      .agg(["count","mean","median","min","max"])
)

        count       mean  median    min    max
Rating                                        
1         226  34.561195  34.770  10.40  59.64
2         196  34.810918  36.215  10.02  59.95
3         203  34.692020  33.780  10.16  59.99
4         179  36.093296  37.800  10.01  59.45
5         196  35.374490  36.900  10.00  59.92


In [18]:
# ---- Price Analysis by Price Band---
print(
    df.groupby("PriceBand", observed=True)["Price"]
      .agg(["count","mean"])
)

                  count       mean
PriceBand                         
Budget (<£20)       196  14.798571
Mid (£20-35)        288  27.119618
Premium (£35-50)    318  42.403648
Luxury (£50+)       198  54.924343


In [19]:
# ----Top Expensive Books---
print("\nTop 10 Most Expensive Books")

print(
    df.nlargest(10,"Price")[["Title","Price","Rating"]]
)


Top 10 Most Expensive Books
                                                 Title  Price  Rating
648                 The Perfect Play (Play by Play #1)  59.99       3
617                  Last One Home (New Beginnings #1)  59.98       3
860                   Civilization and Its Discontents  59.95       2
560                     The Barefoot Contessa Cookbook  59.92       5
366                          The Diary of a Young Girl  59.90       3
657  The Bone Hunters (Lexy Vaughan & Steven Macaul...  59.71       3
133  Thomas Jefferson and the Tripoli Pirates: The ...  59.64       1
387                      Boar Island (Anna Pigeon #19)  59.48       3
393                          The Improbability of Love  59.45       1
549  The Man Who Mistook His Wife for a Hat and Oth...  59.45       4


In [20]:
# ---- Cheapest Books--

print("\nTop 10 Cheapest Books")

print(
    df.nsmallest(10,"Price")[["Title","Price","Rating"]]
)


Top 10 Cheapest Books
                                                 Title  Price  Rating
638                         An Abundance of Katherines  10.00       5
501                              The Origin of Species  10.01       4
716  The Tipping Point: How Little Things Can Make ...  10.02       2
84                                            Patience  10.16       3
302                               Greek Mythic History  10.23       5
558  The Fellowship of the Ring (The Lord of the Ri...  10.27       2
479                                  History of Beauty  10.29       4
242  The Lucifer Effect: Understanding How Good Peo...  10.40       1
434  NaNo What Now? Finding your editing process, r...  10.41       4
274                                       Pet Sematary  10.56       3


In [21]:
# ---Average Price
print(f"Average Price : £{df['Price'].mean():.2f}")

print(f"Median Price : £{df['Price'].median():.2f}")

print(f"Maximum Price : £{df['Price'].max():.2f}")

print(f"Minimum Price : £{df['Price'].min():.2f}")

Average Price : £35.07
Median Price : £35.98
Maximum Price : £59.99
Minimum Price : £10.00


In [22]:
print("\nUnique Values")

for col in ["Rating", "PriceBand", "Availability"]:
    print(f"\n{col}")
    print(df[col].value_counts())


Unique Values

Rating
Rating
1    226
3    203
5    196
2    196
4    179
Name: count, dtype: Int64

PriceBand
PriceBand
Premium (£35-50)    318
Mid (£20-35)        288
Luxury (£50+)       198
Budget (<£20)       196
Name: count, dtype: int64

Availability
Availability
In stock    1000
Name: count, dtype: int64


## Conclusion

The scraped dataset was successfully cleaned and analyzed using Exploratory Data Analysis (EDA) techniques. Missing values, duplicates, and data types were handled appropriately. Statistical analysis, correlation testing, and outlier detection provided meaningful insights into book prices, ratings, and availability.

### Key Findings

- Most books belong to the Budget and Mid-price categories.
- The majority of books are available in stock.
- Price and rating show only a weak correlation.
- A few expensive books were identified as outliers.
- The cleaned dataset is now ready for visualization.
## Future Improvements

- Perform feature engineering for advanced analysis.
- Analyze relationships with additional statistical techniques.
- Use larger datasets for more comprehensive insights.